# Healthcare Agent Learning Lab 3/10

## Human approval as an enforced gate

**Suggested time:** 20 minutes  
**Demonstration type:** Scripted or live agent + deterministic approval gate

### Learning objectives

- Distinguish a prompt request from a code-enforced approval boundary.
- Identify the evidence a reviewer needs before acting.
- Explain why approval identity and auditability matter.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** show the safety principle from the white paper *live*: a consequential action is **enforced by code to require human approval**, not left to the model's judgment.

Same loop as NB 2, but now the agent has a `change_order` tool. That tool is **gated**: the loop pauses and asks a human before anything happens.


> ### In plain terms
> Some actions are too consequential to leave to the model's judgment — like changing a drug dose. This shows the safe pattern: the risky action is **stopped by the code** and needs a **human to approve** it, and every decision is written to a log.
>
> **The point:** safety is built into the wiring, not politely requested in the prompt.
>
> **What you'll see:** the loop runs as before, but watch for the `>> GATE` line — that's the code pausing to ask a human before anything can change.


In [ ]:

import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages):
    hist = " ".join(m["content"] for m in messages if m["role"]!="system")
    if "get_latest_inr" not in hist:
        return json.dumps({"thought":"Read the latest INR.","action":"get_latest_inr","action_input":{}})
    if "change_order" not in hist:
        # The agent PROPOSES a consequential action -> the gate must intercept it.
        return json.dumps({"thought":"INR 4.2 is high; I propose reducing the warfarin dose.",
                           "action":"change_order","action_input":{"order":"warfarin 5 mg -> 3 mg daily"}})
    return json.dumps({"thought":"Deferring to the clinician's decision.","action":"final",
                       "action_input":{"summary":"Consequential action was gated to a human."}})


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### Tools, with one of them gated
`change_order` is a *consequential* action. `GATED` lists which tools require approval. This is the deterministic gate: independent of what the model decides, the order cannot change without a human.


In [ ]:
PATIENT={"inr":4.2,"range":"2.0-3.0","medications":["warfarin 5 mg daily"]}
AUDIT=[]
def _text_arg(a, *names):
    for n in names:
        if a.get(n): return a[n]
    for v in a.values():
        if isinstance(v, str) and v.strip(): return v
    return ""
def get_latest_inr(**_): return {"inr":PATIENT["inr"],"reference_range":PATIENT["range"]}
def change_order(**a):                       # accepts whatever arg name the model uses
    order=_text_arg(a, "order","new_order","change","dose","medication")
    if not order:                            # model didn't name a concrete order -> change nothing
        return {"changed_to": None, "note": "no explicit order given; nothing changed"}
    PATIENT["medications"]=[order]; return {"changed_to":order}
TOOLS={"get_latest_inr":get_latest_inr,"change_order":change_order}
GATED={"change_order"}   # <-- consequential actions require human approval

# For non-interactive runs, set AUTO to "approve"/"deny"; leave None to be asked live.
AUTO = os.environ.get("GATE_AUTO")   # e.g. export GATE_AUTO=deny
def human_approves(action, args):
    prompt=f"\n  >> GATE: agent wants to {action}({args}). Approve? [y/N] "
    if AUTO is not None:
        decision = AUTO.lower().startswith("a"); print(prompt+("y" if decision else "N")+"  (AUTO)"); return decision
    try:    return input(prompt).strip().lower().startswith("y")
    except Exception: print(prompt+"N  (no TTY -> deny by default)"); return False


### The gated loop


In [ ]:
SYSTEM=("You are a clinical assistant. Output ONLY JSON {thought, action, action_input}. "
        "Actions: " + str(list(TOOLS)+["final"]) + ". If the INR is out of range, PROPOSE exactly one "
        'change_order whose action_input is {"order": "<explicit new medication order>"}, for example '
        '{"order": "warfarin 5 mg -> 3 mg daily"}. A human will decide. Then action=final.')
def parse(txt):
    """Parse the first JSON object and tolerate surrounding model commentary."""
    t = re.sub(r"^```[a-z]*|```$", "", txt.strip(), flags=re.M).strip()
    try:
        value = json.loads(t)
    except json.JSONDecodeError:
        start = t.find("{")
        if start < 0:
            raise ValueError(f"Model did not return a JSON object: {t[:200]!r}")
        value, _end = json.JSONDecoder().raw_decode(t[start:])
    if not isinstance(value, dict):
        raise ValueError("Model action must be a JSON object.")
    allowed_fields = {"thought", "rationale", "action", "action_input"}
    unexpected = set(value) - allowed_fields
    if unexpected:
        raise ValueError(f"Unexpected action fields: {sorted(unexpected)}")
    action = value.get("action")
    if not isinstance(action, str) or not action:
        raise ValueError("Model action must contain a non-empty string 'action'.")
    action_input = value.get("action_input", {})
    if not isinstance(action_input, dict):
        raise ValueError("Model action_input must be a JSON object.")
    value["action_input"] = action_input
    available_tools = set(globals().get("TOOLS", {}))
    if available_tools and action not in available_tools | {"final"}:
        allowed = sorted(available_tools | {"final"})
        raise ValueError(f"Action {action!r} is not allowed; expected one of {allowed}.")
    if action in {"change_order", "write_flag"}:
        has_text = any(isinstance(item, str) and item.strip() for item in action_input.values())
        if not has_text:
            raise ValueError(f"Action {action!r} requires a non-empty textual argument.")
    if action == "draft_followup" and not action_input:
        raise ValueError("Action 'draft_followup' requires a non-empty structured payload.")
    return value

msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":"Review this patient."}]
for step in range(6):
    act=parse(chat(msgs))
    a=act.get("action","final")
    print(f"[{step}] PLAN: {act.get('thought','')}")
    if a=="final": print(f"    DONE: {act.get('action_input',{}).get('summary','')}"); break
    args=act.get("action_input",{})
    if a not in TOOLS: print(f"    (unknown action {a!r}; stopping)"); break
    if a in GATED:                       # <-- the gate fires here
        ok=human_approves(a,args)
        AUDIT.append({"action":a,"args":args,"approved":ok})
        obs=TOOLS[a](**args) if ok else {"blocked":"human denied — no change made"}
    else:
        obs=TOOLS[a](**args)
    print(f"    OBS : {obs}")
    msgs.append({"role":"assistant","content":json.dumps(act)})
    msgs.append({"role":"user","content":f"[{a}] observation: {json.dumps(obs)}"})

print("\nCurrent medications:", PATIENT["medications"])
print("Audit log (who decided what):", AUDIT)


### Takeaway
The consequential action never executed on the model's say-so — a human decided, and the decision was **logged**. That gate + audit trail is the difference between an assistant and a liability, and it's exactly the *safety-as-architecture* point: safety is enforced by the code path, not requested in a prompt.

*Try:* `export GATE_AUTO=approve` then re-run to see the approved path; and point the backend at a real open-weight model to watch a genuine agent hit the same gate.


## Exercise

No programming is required for this section.

**Activity:** Assume the agent proposes a consequential change. Choose approve, reject, or request more information. List the minimum evidence, appropriate reviewer role, and urgency that should accompany the request.

### Discussion prompts

- Does placing a person in the loop guarantee a safe decision?
- What would make the approval request easy to rubber-stamp?
- Which actions require one reviewer, two reviewers, or no reviewer?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: change the decision and the evidence list. No clinical action is executed.
REVIEW_DECISION = "request_more_information"  # "approve", "reject", or "request_more_information"
EVIDENCE_SHOWN = ["latest INR", "current medication"]
REQUIRED_EVIDENCE = {"latest INR", "current medication", "bleeding assessment", "prescriber identity"}
missing = sorted(REQUIRED_EVIDENCE - set(EVIDENCE_SHOWN))
print("Decision:", REVIEW_DECISION)
print("Missing evidence:", missing or "none")
print("Teaching result:", "Hold the action" if missing or REVIEW_DECISION != "approve" else "Eligible for the simulated gated path")


## Optional technical extension

Extend the audit entry with reviewer identity, timestamp, reason, source evidence, and an immutable proposal identifier. Ensure denial cannot be bypassed with a renamed action.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

The dose proposal is fictional and must not be interpreted as a protocol. The learning target is the enforced control path and the information supplied to the reviewer, not whether the proposed dose is clinically correct.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
